[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gravitational_waves/rendering.ipynb)

# Inside a Closed Universe

This example renders a closed universe, the round three-sphere, from inside by stepping rays of light across a tetrahedral triangulation that fills the sphere. Light in the round three-sphere follows great circles, so every edge arc appears as a straight strut and struts fill every sightline; past a quarter turn from the eye, diverging great circles reconverge toward the antipode and distant struts loom larger again. Each ray crosses one tetrahedron per leg along a straight chart segment and exits through the first facet it meets into the adjacent tetrahedron.

Points on the unit three-sphere are modelled as antivectors with the origin as the pole of the equatorial plane, and every place carries a smooth left-isoclinic frame motor `frame(place)` that carries the origin there by turning equally in two orthogonal planes. Because free geodesics hold constant heading in those frames, a ray from `place` rolls along its great circle by `frame(place) * (moving(heading) * (-length / 2)).exp()`. The term ***simplex gradient*** here means the plane through a corner's opposite facet scaled to pair to one at that corner, so `gradients & point` reads the corner weights while their sum defines the tetrahedron's chart hyperplane on which every great circle projects to a straight line. Linear gravitational waves and interior crystal gauges appear in [waves.ipynb](waves.ipynb), the projection onto the two-sphere below the Hopf circles appears in [hopf_projection.ipynb](hopf_projection.ipynb), and `examples.relativity.gravitational_waves.web.page()` builds an interactive WebGPU flight in the browser.

In Lie-group and homogeneous-coordinate notation, the unit three-sphere is $\mathrm{SU}(2)$ with its bi-invariant metric, left-invariant frame $e_i(q) = q\,\tau_i$, and constant-heading geodesics $q\exp(t\,\xi)$ for $\xi \in \mathfrak{su}(2)$, while gnomonic projection from the origin maps spherical arcs on each tetrahedral cone onto straight segments in $\mathbb{R}^3$ parameterized by affine barycentric coordinates.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np
from numga import Algebra, NumpyContext
from numga.topology import Topology

from examples.util.animation import show_animation
from examples.relativity.gravitational_waves import render, scenarios

ga = Algebra("x+y+z+w+")
mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within(Algebra("x+y+z+"))                                # Vector+
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Through.derive.dual                                                      # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Translating = ga.gatype((Step, Through))                                        # Point- <- Plane+

# Equatorial plane `w` and the origin, its pole on the three-sphere.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
# Step of the origin across a plane through it; its inverse sends each step to its polar plane as a duality map.
polar = origin.commutator(equator ^ Through).cast(Translating).inverse()        # [] Plane+ <- Point-
moving = equator ^ polar(Step)                                                  # [] Rate+ <- Point-

## The frame at each place

In four dimensions, multiplying a bivector rate by the extrinsic pseudoscalar produces the rate in the orthogonal plane, splitting every rate into an invariant half and a flipped half. Each half turns by equal angles in two orthogonal planes, moving every point of the sphere the same distance along a great circle. Keeping the pseudoscalar-invariant half of `place / origin` with unit scalar part in the other half produces a smooth frame motor `frame(place)` everywhere on the three-sphere.

In [ ]:
pseudoscalar = mv.xyzw                                                          # [] Pseudoscalar+


def frame(places: Point) -> Motor:
    """Smooth left-isoclinic frame motors `Motor+` carry the origin to `places` by turning equally in the great-circle
    plane and its orthogonal complement."""
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## Legs across the tetrahedra

On each tetrahedron's chart hyperplane, every great circle projects to a straight segment and the simplex-gradient corner weights vary linearly along it, so a single straight leg carries each ray across an entire tetrahedron to the facet whose opposite corner weight reaches zero first. Because free headings remain constant in the left-isoclinic frames, the next leg resumes in the neighbouring tetrahedron with the same heading until the ray hits an edge strut or completes half a turn.

In [ ]:
reach = 0.6
walk = '''
struct Tet {
    neighbours: vec4<u32>,
    entries: vec4<u32>,
    gradients: array<Plane, 4>,
}

var<storage, read> tets: array<Tet>;
var<uniform> origin: Point;
var<uniform> moving: Moving;
var<uniform> reach: f32;

struct Leg {
    plane: Plane,
    chart: Point,
    segment: Point,
    length: f32,
    heading: Step,
    tet: u32,
    entry: u32,
}

fn frame(place: Point) -> Motor {
    return ((1.0 + mv.xyzw) * (place / origin) + (1.0 - mv.xyzw)) * 0.5;
}

fn next_leg(place: Point, heading: Step, tet: u32, entry: u32, time: f32) -> Leg {
    let unit = tets[tet].gradients[0] + tets[tet].gradients[1] + tets[tet].gradients[2] + tets[tet].gradients[3];
    let chart = place / (unit & place);
    let reach_end = (frame(place) * (moving(heading) * (-0.5 * reach)).exp()) >> origin;
    let segment = reach_end / (unit & reach_end) - chart;
    var share = 1.0;
    var exit = 4u;
    for (var corner = 0u; corner < 4u; corner++) {
        let change = tets[tet].gradients[corner] & segment;
        if (corner != entry && change < 0.0) {
            let run_out = max(-(tets[tet].gradients[corner] & chart) / change, 0.0);
            if (run_out < share) {
                share = run_out;
                exit = corner;
            }
        }
    }
    let end = (chart + segment * share).normalized();
    var this_leg = Leg(unit, chart, segment * share, acos(clamp(place.dot(end), -1.0, 1.0)), heading, tet, 4u);
    if (exit < 4u) {
        this_leg.tet = tets[tet].neighbours[exit];
        this_leg.entry = tets[tet].entries[exit];
    }
    return this_leg;
}
'''

## A flight through the 600-cell

Within the 600-cell, thirty regular tetrahedra stacked face to face form a closed Boerdijk–Coxeter helix whose central great-circle axis threads an unobstructed tunnel around the sphere. Placing the camera motor on that axis and advancing along its forward axis carries the eye once around the closed universe and back to its start. Ray-tracing the ninety frames takes about five seconds.

In [ ]:
n_flight_frames = 90
duration_ms = 40

In [ ]:
cell_topology, cell_vertices = scenarios.six_hundred_cell()
rails, axis_point, axis_direction, side_direction = scenarios.helix(cell_vertices)
circles = scenarios.great_circles(cell_topology, cell_vertices)


def simplex_gradients(topology: Topology, vertices: Point) -> Plane:
    """Simplex gradients `[4] Plane+[T]` pair to one at each tetrahedron corner and vanish on the opposite facet."""
    corners = topology.incidence[3, 0] * vertices                               # [4] Point-[T]
    opposite = topology.off[3, 0] * vertices                                    # [3, 4] Point-[T]
    facets = opposite[0] & opposite[1] & opposite[2]                            # [4] Plane-[T]
    return facets / (facets & corners)


def camera_aim(placing: Motor, forward: Point, up: Point) -> Motor:
    """Camera motor `[] Motor+` composes `placing` with rotors aligning the forward and up steps at the eye."""
    aiming = (forward / mv.z.dual()).square_root()                              # [] Motor+
    rolling = (up / (aiming >> mv.y.dual())).square_root()                      # [] Motor+
    return placing * rolling * aiming


cell_gradients = simplex_gradients(cell_topology, cell_vertices)                # [4] Plane+[T]
flight_start = camera_aim(frame(axis_point), axis_direction, side_direction)    # [] Motor+
forward_rate = moving(mv.z.dual())                                              # [] Rate+
distances = np.linspace(0, 2 * np.pi, n_flight_frames, endpoint=False)          # [n_flight_frames]
flight = flight_start * (forward_rate * (-distances / 2)).exp()                 # [n_flight_frames] Motor+
cell_walk_values = dict(gradients=cell_gradients, origin=origin, moving=moving, reach=reach)
show_animation(render.polygon_view(walk, cell_walk_values, cell_topology, cell_vertices, flight, circles),
               "gravitational_waves_flight", duration_ms);

Seventy-two distinct colours distinguish the great-circle decagons of the 600-cell so each circle's colour continues straight through every vertex; beyond a quarter turn from the eye, struts grow larger again as sightlines converge on the antipode.

## The helix seen from outside

Viewed from nearly a quarter turn off the helix axis, thirty stacked tetrahedra wind three decagon rails around the central tunnel. Translating the camera along the axis plane leaves the orthogonal circle invariant, so the camera glides parallel to the helix without orbiting around it while the three rails corkscrew past.

In [ ]:
n_pan_frames = 60
pan_distance = 1.02
pan_roll = 0.36

In [ ]:
station = frame(axis_point) * (moving(side_direction) * (-pan_distance / 2)).exp()   # [] Motor+
pan_start = camera_aim(station, -side_direction, axis_direction) * (mv.xy * (pan_roll / 2)).exp()   # [] Motor+
axis_rate = frame(axis_point) >> moving(axis_direction)                         # [] Rate+
angles = np.linspace(0, 2 * np.pi, n_pan_frames, endpoint=False)                # [n_pan_frames]
pan = (axis_rate * (-angles / 2)).exp() * pan_start                             # [n_pan_frames] Motor+
show_animation(render.helix_view(walk, cell_walk_values, cell_topology, cell_vertices, pan, rails),
               "gravitational_waves_helix", duration_ms);

Three distinct colours highlight the decagon rails of the Boerdijk–Coxeter helix, pale struts mark the rungs joining them, and faint ghosts trace the surrounding 600-cell.

## A flight through an irregular triangulation

The chart-leg shader traverses any tetrahedral triangulation of the three-sphere unchanged. Combining the 600-cell vertices with relaxed icosahedral orbits of random seeds produces an irregular triangulation whose edges follow no shared great circles; struts currently containing the eye are omitted so near crossings never block the view. Relaxing the triangulation and tracing the flight takes about ten seconds.

In [ ]:
irregular_topology, irregular_vertices = scenarios.irregular_three_sphere()
irregular_gradients = simplex_gradients(irregular_topology, irregular_vertices)   # [4] Plane+[T]
middle = (irregular_topology.incidence[3, 0].mean(axis=0) * irregular_vertices).batch()[0].normalized()   # [] Point-
irregular_flight = frame(middle) * (forward_rate * (-distances / 2)).exp()      # [n_flight_frames] Motor+
show_animation(render.plain_view(walk, dict(gradients=irregular_gradients, origin=origin, moving=moving, reach=reach),
                                 irregular_topology, irregular_vertices, irregular_flight),
               "gravitational_waves_irregular", duration_ms);

Neutral struts outline every edge of the relaxed irregular triangulation as the camera completes one closed great-circle loop back to its starting tetrahedron.

In [ ]:
# checks
misses = (frame(cell_vertices) >> origin) - cell_vertices                       # Point-[V]
assert (misses.norm() < 1e-10).all()
once_round = (flight_start * (forward_rate * -np.pi).exp()) >> origin           # [] Point-
back = once_round - (flight_start >> origin)                                    # [] Point-
assert back.norm() < 1e-10
assert np.isin(cell_topology.elements[3], rails).all(axis=-1).sum() == 30